# Day 3: Density regressions and implications for Morocco

ECU Summer School of Economic Complexity. UM6P Rabat. Afternoon Session 3.

Continuing from yesterday - density ranks how near each product sits to your country's basket, and we were careful not to claim more: whether nearby products actually get entered is an empirical question. Today we run that empirical test.

The steps: compute density from an earlier year so that it is measured before the outcome, fit the cross-section to see how much of the world's current export pattern density alone can explain, mark which products each country entered since then, test whether entry concentrates where density was high, and read the results the way you would defend them in front of a senior policymaker.

`COUNTRY`, `COMPARATORS`, and the two years at the top drive everything. On Google Colab, save your own copy before anything else: File menu, "Save a copy in Drive".

## 0. Setup

Same data packet as Days 1 and 2, nothing new to download. Two new tools join `ecomplexity`: `pyfixest` runs regressions with fixed effects, and `maketables` turns fitted models into a readable table.

In [ ]:
import urllib.request
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# The complexity engine, as yesterday: the development branch of py-ecomplexity,
# which accepts a custom Mcp matrix, installed on the spot if missing.
try:
    from ecomplexity import ecomplexity
except ModuleNotFoundError:
    import subprocess

    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--system",
            "git+https://github.com/harvard-growth-lab/py-ecomplexity@develop",
        ],
        check=True,
    )
    from ecomplexity import ecomplexity

# Today's additions: fixed-effects regressions (pyfixest) and regression tables
# (maketables, which renders through great-tables). Same self-healing install.
try:
    import pyfixest as pf
    from maketables import ETable
except ModuleNotFoundError:
    import subprocess

    subprocess.run(
        ["uv", "pip", "install", "--system", "pyfixest", "maketables", "great-tables"],
        check=True,
    )
    import pyfixest as pf
    from maketables import ETable

warnings.filterwarnings("ignore", category=FutureWarning)

# --- The ONE place you edit to make this notebook about your own country. ------
# Codes are ISO 3166-1 alpha-2 (two letters). See reference/units.csv.
COUNTRY = "MA"  # the focus country (Morocco)
COMPARATORS = ["EG", "TN", "KR"]  # Egypt, Tunisia (MENA) + Korea (frontier)
YEAR_T0 = 2012  # density is measured here, BEFORE the outcome
YEAR_T1 = 2023  # entries and exits are observed here
YEAR_EARLY = 2001  # start of the first growth window in section 9
# -------------------------------------------------------------------------------

LOCAL_PACKET = Path("../../data/processed/morocco_data_packet")
DATA_RELEASE = "https://github.com/shreyasgm/ecu-complexity-labs/releases/download/data-v1"

if LOCAL_PACKET.exists():
    PACKET = LOCAL_PACKET.resolve()
else:
    PACKET = Path("morocco_data_packet").resolve()
    for folder in ("exports", "reference"):
        if not (PACKET / folder).exists():
            print(f"Downloading {folder}.zip from the course data release ...")
            zip_file, _ = urllib.request.urlretrieve(f"{DATA_RELEASE}/{folder}.zip")
            with zipfile.ZipFile(zip_file) as zf:
                zf.extractall(PACKET)

# Fail loud on anything missing, naming the exact file.
required = [
    PACKET / "exports" / "outputs.parquet",
    PACKET / "reference" / "fields.csv",
    PACKET / "reference" / "units.csv",
]
for f in required:
    if not f.exists():
        raise FileNotFoundError(
            f"Required file not found: {f}\n"
            "Delete your local morocco_data_packet folder and re-run this cell "
            f"to fetch a fresh copy. Release: {DATA_RELEASE}"
        )

exports = pd.read_parquet(PACKET / "exports" / "outputs.parquet")
fields = pd.read_csv(PACKET / "reference" / "fields.csv")
units = pd.read_csv(PACKET / "reference" / "units.csv")

# The packet stores measures as float32 to keep the download small. Upcast
# before computing, or identity checks fail on the last digits.
float32_cols = exports.select_dtypes("float32").columns
exports[float32_cols] = exports[float32_cols].astype("float64")

prod_name = fields.set_index("Field ID")["Field Name"]
PKG_COLS = {"time": "time", "loc": "loc", "prod": "prod", "val": "val"}
print(f"Packet: {PACKET}")
print(f"Focus country: {COUNTRY} | window: {YEAR_T0} -> {YEAR_T1}")

## 1. Compute complexity metrics in a function

Today we need the whole Day 2 pipeline at several points in time, so we wrap it in a function of the year: export matrix, RCA, the continuous M-hat, and one package call that returns all the measures. Nothing in the body is new.

In [ ]:
def complexity_for_year(year: int) -> dict:
    """Run the Day 1 to Day 2 chain for one year and return every object we use."""
    year_long = exports.query("Period == @year")[["Unit", "Field ID", "Outputs"]]
    X = year_long.pivot_table(index="Unit", columns="Field ID", values="Outputs", fill_value=0.0)
    # Early years have a few countries or products with zero recorded trade;
    # drop them or the shares divide by zero and the package warns about NaNs.
    X = X.loc[X.sum(axis=1) > 0, X.sum(axis=0) > 0]
    shares = X.div(X.sum(axis=1), axis=0)
    world_share = X.sum(axis=0) / X.sum().sum()
    rca = shares.div(world_share, axis=1)
    mcp = (rca > 1).astype(float)  # the binary basket, for events and highlights
    m_hat = rca / (1 + rca)  # the continuous presence matrix, for the measures

    mhat_long = m_hat.stack().rename("val").reset_index()
    mhat_long.columns = ["loc", "prod", "val"]
    mhat_long["time"] = year

    # COMPLETION exercise: yesterday's hard-won lesson. The package must take our
    # M-hat AS the presence matrix instead of recomputing RCA on it.
    #
    # PROMPT: add one line here defining `presence_rule`, set to the
    # presence_test value that tells the package the val column is already the
    # presence matrix. Until it exists, the call below stops with a NameError.

    # TODO(core): complete the one missing line below.

    cdata = ecomplexity(
        mhat_long,
        PKG_COLS,
        presence_test=presence_rule,
        check_logsupermodularity=False,
        verbose=False,
    )
    return {"year": year, "X": X, "rca": rca, "mcp": mcp, "m_hat": m_hat, "cdata": cdata}


res_t1 = complexity_for_year(YEAR_T1)
diversity_t1 = res_t1["mcp"].sum(axis=1)
print(f"{COUNTRY} diversity in {YEAR_T1}:", int(diversity_t1.loc[COUNTRY]))

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. Yesterday's fix-the-bug: without this argument the package treats the weights as raw export values, recomputes RCA, and binarizes. The self-check below catches exactly that.

Hint 2. The docstring says presence_test="manual" means M_cp is taken as given from the value column. So `presence_rule = "manual"`.
</details>

In [ ]:
# Self-check: the Day 2 handshake. Same data and method must give yesterday's
# numbers, and the package must have used the continuous weights.
_mcp_col = res_t1["cdata"]["mcp"].dropna()
assert _mcp_col.nunique() > 100, (
    "the returned mcp column is binary, so the package recomputed presence from "
    "your values. Set presence_rule so M-hat is used as given."
)
_dens_ma = res_t1["cdata"].query("loc == @COUNTRY").set_index("prod")["density"]
_in_b = _dens_ma[res_t1["mcp"].loc[COUNTRY] == 1].mean()
_out_b = _dens_ma[res_t1["mcp"].loc[COUNTRY] == 0].mean()
assert _in_b > _out_b, (
    "density inside the basket must exceed density outside it, as on Day 2. "
    "If not, the loc/prod columns are probably swapped."
)
if COUNTRY == "MA" and YEAR_T1 == 2023:
    assert int(diversity_t1.loc[COUNTRY]) == 129, (
        "Morocco's 2023 diversity was 129 on Days 1 and 2 and must be 129 today."
    )
    assert abs(_in_b - 0.220) < 0.015 and abs(_out_b - 0.187) < 0.015, (
        "Morocco's 2023 in/out-basket mean densities were 0.220 and 0.187 "
        "yesterday and must reproduce today."
    )
print(f"OK: pipeline reproduced ({_in_b:.3f} in-basket vs {_out_b:.3f} outside).")

## 2. Rewind to the earlier year

A prediction has to be made before the outcome. If we scored products by density today and then checked which products are in today's basket, that would be circular. So the whole pipeline runs again for an earlier year, and everything we use as a predictor comes from there. Before running the next cell, make a guess: was your country's basket bigger or smaller eleven years ago, and which of its flagship exports today were not yet competitive then?

Explore the data and check your guess

In [ ]:
res_t0 = complexity_for_year(YEAR_T0)

# Align the two years on the countries and products present in both, so every
# comparison below is cell for cell.
common_c = res_t0["rca"].index.intersection(res_t1["rca"].index)
common_p = res_t0["rca"].columns.intersection(res_t1["rca"].columns)
mcp0 = res_t0["mcp"].loc[common_c, common_p].astype(bool)
mcp1 = res_t1["mcp"].loc[common_c, common_p].astype(bool)

print(f"aligned: {len(common_c)} countries x {len(common_p)} products")
print(
    f"{COUNTRY} diversity: {int(mcp0.loc[COUNTRY].sum())} in {YEAR_T0} "
    f"-> {int(mcp1.loc[COUNTRY].sum())} in {YEAR_T1}"
)

# Which of the country's ten biggest current exports were not competitive at t0?
top_now = res_t1["X"].loc[COUNTRY, common_p].nlargest(10)
was_out = [prod_name.get(p, p) for p in top_now.index if not mcp0.loc[COUNTRY, p]]
print(f"top-10 {YEAR_T1} exports that had no RCA in {YEAR_T0}: {was_out}")

In [ ]:
# Self-check: the alignment kept the world intact.
assert len(common_c) > 150 and len(common_p) > 800, (
    "the aligned matrices lost too many countries or products. Both years should "
    "cover nearly the full sample; check the Period filter in the function."
)
assert mcp0.shape == mcp1.shape, "the two years must be aligned to the same shape."
print("OK: both years aligned cell for cell.")

## 3. The cross-section: density fits the world as it is

Before asking whether density predicts change, the morning's two-stage approach asks a simpler question first: how much of the world's export pattern, as it stands, does density explain? This is the cross-sectional regression from Lecture 5, run on one year of data: the log of RCA against the log of relatedness density - each row represents one country and product. The lecture's table adds a second regressor built from the "country space" (countries like you export this product); we stay with the product space density you built yesterday, which carries most of the fit on its own.

In [ ]:
rca0 = res_t0["rca"].loc[common_c, common_p]

# Product space density for the same year, reshaped to a matrix so it aligns
# with the RCA matrix cell for cell.
ps_density = (
    res_t0["cdata"].pivot(index="loc", columns="prod", values="density").loc[common_c, common_p]
)

# The cross-section frame: one row per cell with positive exports, in logs as in
# the lecture's table (its RpCA variant normalizes by population; ours is the
# export RCA you have used all week, same design).
cs_frame = pd.DataFrame(
    {
        "log_rca": np.log(rca0.stack().where(lambda s: s > 0)),
        "log_ps": np.log(ps_density.stack()),
    }
).dropna()
cs_frame.index.names = ["loc", "prod"]
cs_frame = cs_frame.reset_index()
print(f"{len(cs_frame):,} country-product cells with positive exports in {YEAR_T0}")

In [ ]:
m_cross = pf.feols("log_rca ~ log_ps", data=cs_frame, vcov={"CRV1": "loc"})

ETable(
    m_cross,
    labels={
        "log_rca": f"RCA (log), {YEAR_T0}",
        "log_ps": "Product space density (log)",
    },
    notes=(
        "Cross-section of country-product cells with positive exports. "
        "Standard errors clustered by country."
    ),
).make("gt")

In [ ]:
# Self-check: the lecture's qualitative pattern, not its numbers.
_b_cross = float(m_cross.coef()["log_ps"])
assert _b_cross > 0, (
    "density must relate positively to RCA in the cross-section; a negative "
    "sign points at an inverted log or a broken join."
)
assert abs(m_cross.tstat()["log_ps"]) > 5, (
    "with this many cells the cross-sectional fit should be precisely estimated."
)
assert m_cross._r2 > 0.1, (
    f"density alone should explain a visible share of the cross-section "
    f"(got R2 {m_cross._r2:.2f}); much lower suggests the density join failed."
)
print(
    f"OK: a 1% higher density lines up with a {_b_cross:.2f}% higher RCA "
    f"(R2 {m_cross._r2:.2f} from density alone)."
)

Density explains a visible share of who exports what, before any country or product adjustment and with only revealed co-location as input, which is why the lecture treats it as a summary of a country's productive ecosystem. This regression also cannot answer the question we actually care about, because in a single year density and RCA are two views of the same matrix. The real test is dynamic, and everything that follows sets it up.

## 4. Mark entries and exits

An entry is a product the country did not export competitively at the start of the window but does at the end. This is one boolean line, and it is the object every regression today explains. Exits are the mirror image, and they matter just as much: baskets churn, and a serious analysis counts both directions.

There are many ways to mark entries and exits, but this is the most intuitive: a cell is an entry when the country was NOT in the product at the start of the window but is in it at the end. In practice, you would have a more sophisticated definition.

In [ ]:
# TODO(core): mark entries. A cell is an entry when the country was NOT in the
# product at YEAR_T0 (mcp0 is False) AND is in it at YEAR_T1 (mcp1 is True).
#
# PROMPT: combine mcp0 and mcp1 with ~ (not) and & (and). One line, no loops.

# TODO(core): write your code here
raise NotImplementedError()

exited = mcp0 & ~mcp1  # worked for you: the mirror image

n_candidates = int((~mcp0).values.sum())
n_incumbents = int(mcp0.values.sum())
entry_rate = appeared.values.sum() / n_candidates
exit_rate = exited.values.sum() / n_incumbents
print(f"candidate cells (no RCA at {YEAR_T0}): {n_candidates:,}")
print(f"entries: {int(appeared.values.sum()):,} ({entry_rate:.1%} of candidates)")
print(f"exits:   {int(exited.values.sum()):,} ({exit_rate:.1%} of incumbents)")
print(
    f"{COUNTRY}: {int(appeared.loc[COUNTRY].sum())} entries, {int(exited.loc[COUNTRY].sum())} exits"
)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. You need cells that are False in mcp0 and True in mcp1. Boolean DataFrames combine elementwise, exactly like the matrices on Day 1.

Hint 2. `~mcp0` flips the start-of-window matrix, so `~mcp0 & mcp1` is True exactly where a product was absent then and present now.
</details>

In [ ]:
# Self-check: the event definitions must be internally consistent.
assert not (appeared & mcp0).any().any(), (
    "an entry cannot happen in a product the country already exported at t0. "
    "The definition needs ~mcp0, not mcp0."
)
assert not (appeared & exited).any().any(), "no cell can be both an entry and an exit."
assert 0.01 < entry_rate < 0.15, (
    f"the pooled entry rate should be a few percent (got {entry_rate:.1%}). "
    "Much higher usually means the t0 condition is missing."
)
assert 0.1 < exit_rate < 0.6, (
    f"the pooled exit rate should be substantial (got {exit_rate:.1%}); "
    "RCA status churns more than most people expect."
)
print("OK: entries and exits marked consistently.")

Keep that exit rate in mind. Roughly a third of the products a country exports competitively in one year are gone from its basket eleven years later. Comparative advantage is a moving target, which is exactly why a static opportunity list is not enough and we need to know what predicts the movement.

## 5. Test table + find the bug

### Fix the broken cell

The cell below assembles everything into one regression table: one row per country and product, with the outcome events and the density score. As shipped, it fills the density column from `YEAR_T1`, the same year the outcome is measured. That version runs without an error and produces a strong-looking result, and it is wrong: entering a product mechanically raises the density around it, so measuring density after the fact bakes the answer into the question. Repair the cell so density comes from `YEAR_T0`, the year before any of the outcomes happened.

In [ ]:
# One row per aligned country-product cell.
frame = pd.DataFrame(
    {
        "incumbent": mcp0.stack().astype(int),
        "appeared": appeared.stack().astype(int),
        "exited": exited.stack().astype(int),
        "x0": res_t0["X"].loc[common_c, common_p].stack(),
        "x1": res_t1["X"].loc[common_c, common_p].stack(),
    }
)
frame.index.names = ["loc", "prod"]
frame = frame.reset_index()
key = pd.MultiIndex.from_frame(frame[["loc", "prod"]])

dens_t0 = res_t0["cdata"].set_index(["loc", "prod"])["density"]
dens_t1 = res_t1["cdata"].set_index(["loc", "prod"])["density"]

frame["density"] = dens_t1.reindex(key).values  # first attempt: outcome-year density

# TODO(core): the line above predicts the present with the present. Overwrite
# the density column with the YEAR_T0 values so the predictor comes strictly
# before the outcome.
#
# PROMPT: same line, other series: fill from dens_t0 instead of dens_t1.

# TODO(core): write your code here
raise NotImplementedError()

# Standardize once so every coefficient below reads "per one standard deviation
# of density", which is easier to compare across models than raw units.
frame["density_sd"] = (frame["density"] - frame["density"].mean()) / frame["density"].std()
frame.head(3)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. The self-check compares your density column against the package output for `YEAR_T0`. If it fails, your column still holds `YEAR_T1` values.

Hint 2. Replace `dens_t1` with `dens_t0` in the last assignment, keeping the same `.reindex(key).values` pattern so the rows stay aligned.
</details>

In [ ]:
# Self-check: the predictor must predate the outcome, cell for cell.
_expected = dens_t0.reindex(key).values
assert frame["density"].notna().all(), (
    "every aligned cell has a t0 density; missing values mean the reindex key "
    "does not match the loc/prod columns."
)
assert np.allclose(frame["density"].values, _expected), (
    f"the density column must hold the {YEAR_T0} values. Using {YEAR_T1} density "
    "regresses the outcome on itself: entering a product raises the density "
    "around it, so the timing is the whole point."
)
print(f"OK: density measured in {YEAR_T0}, outcomes measured in {YEAR_T1}.")

## 6. The Principle of Relatedness

The morning stated the principle: economies diversify preferentially into activities related to what they already do. Before any regression, look at it raw. Split the candidate products (no RCA at t0) into ten equal groups by their t0 density and compute the share of each group that had been entered by t1.

### Predict before running

How much more often does the top density decile get entered compared with the bottom decile: about 3x, about 12x, or about 30x? Commit to a number before running the cell.

In [ ]:
cand = frame.query("incumbent == 0").copy()
cand["decile"] = pd.qcut(cand["density"], 10, labels=False) + 1  # 1 = least related

# TODO(core): the entry rate per decile: group the candidates by decile and take
# the mean of the appeared dummy (the mean of a 0/1 variable IS the rate).
#
# PROMPT: one groupby line ending in .mean().

# TODO(core): write your code here
raise NotImplementedError()

ratio = entry_by_decile.iloc[-1] / entry_by_decile.iloc[0]
print((entry_by_decile * 100).round(2).rename("entry rate (%)").to_string())
print(f"top decile / bottom decile = {ratio:.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(entry_by_decile.index, entry_by_decile * 100, color="#5b9bd5", edgecolor="white")
ax.bar(
    entry_by_decile.index[-1],
    entry_by_decile.iloc[-1] * 100,
    color="#ee3e4c",
    edgecolor="white",
)
ax.set_xticks(entry_by_decile.index)
ax.set_xlabel(f"Relatedness density in {YEAR_T0} (deciles of candidate products)")
ax.set_ylabel(f"Share entered by {YEAR_T1} (%)")
ax.set_title(
    f"Products near the basket get entered more often ({len(cand):,} candidate cells, all countries)"
)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. `cand` already has a decile column. You want, within each decile, the average of `appeared`.

Hint 2. `cand.groupby("decile")["appeared"].mean()` returns exactly the ten rates, indexed by decile.
</details>

In [ ]:
# Self-check: the gradient the principle predicts.
assert len(entry_by_decile) == 10, "ten deciles in, ten entry rates out."
assert entry_by_decile.max() <= 1, (
    "these are rates (means of a 0/1 dummy) and must lie in [0, 1]; a value "
    "above 1 means the groupby summed instead of averaged."
)
assert entry_by_decile.iloc[-1] > 2 * entry_by_decile.iloc[0], (
    "the top density decile should be entered at least twice as often as the "
    "bottom one; a flat profile means the density column is broken (or the "
    "Principle of Relatedness just failed in our data, which would be news)."
)
_rho_dec = spearmanr(entry_by_decile.index, entry_by_decile.values).statistic
assert _rho_dec > 0.7, (
    f"entry rates should rise with the density decile (Spearman {_rho_dec:.2f}). "
    "One noisy decile is fine; a non-monotone mess is not."
)
print(f"OK: entry rises with density ({ratio:.1f}x from bottom to top decile).")

This is the same figure the Principle of Relatedness literature shows for products, industries, technologies, and research areas, drawn from your own pipeline. The gradient is real, and it is also not yet a regression: rich diversified countries enter more products everywhere, and popular products get entered by everyone. The next section accounts for both.

## 7. Density regressions

The bar chart pools everything, so part of its gradient could just be that diversified countries enter more and common products attract everyone. The fix is fixed effects. A country fixed effect absorbs everything constant about the country in this window (size, income, how many products it enters overall), and a product fixed effect absorbs everything constant about the product (how entry-prone it is worldwide). What survives is within-country, within-product variation: is the same country more likely to enter a product when it sits closer to its basket than the country's overall entry activity and the product's overall popularity would predict?

We run four regressions on the same table. Entry appears twice, once pooled (no fixed effects, so it is the bar chart in one number) and once within (both fixed effects on). The entry and exit columns are linear probability models, ordinary regressions on a 0/1 outcome, so each coefficient reads directly as a change in probability. Note that in practice, you would use logit / probit models. Exit and export growth follow with fixed effects, and both control for the initial export level: a product a country barely exports is easier to lose, and small export lines grow faster mechanically. Growth is measured on the intensive margin, meaning how much existing export lines grew as opposed to new ones appearing, for cells that already exported something at t0, transformed with asinh, which behaves like a log but tolerates collapses to zero. Standard errors are clustered by country, since outcomes within a country move together. Entry has no initial-level control on purpose, since a product you do not yet make has no meaningful initial value.

In [ ]:
# Growth on the intensive margin: asinh difference tolerates x1 = 0 (a collapse),
# unlike log growth. ihs in the variable names is the inverse hyperbolic sine,
# numpy's arcsinh. We control for the initial level, the usual convergence term.
frame["growth_ihs"] = np.arcsinh(frame["x1"]) - np.arcsinh(frame["x0"])
frame["ihs_x0"] = np.arcsinh(frame["x0"])

m_entry_pooled = pf.feols(
    "appeared ~ density_sd", data=frame.query("incumbent == 0"), vcov={"CRV1": "loc"}
)
m_entry = pf.feols(
    "appeared ~ density_sd | loc + prod", data=frame.query("incumbent == 0"), vcov={"CRV1": "loc"}
)
m_exit = pf.feols(
    "exited ~ density_sd + ihs_x0 | loc + prod",
    data=frame.query("incumbent == 1"),
    vcov={"CRV1": "loc"},
)
m_growth = pf.feols(
    "growth_ihs ~ density_sd + ihs_x0 | loc + prod",
    data=frame.query("x0 > 0"),
    vcov={"CRV1": "loc"},
)

ETable(
    [m_entry_pooled, m_entry, m_exit, m_growth],
    labels={
        "appeared": f"Entered by {YEAR_T1}",
        "exited": f"Exited by {YEAR_T1}",
        "growth_ihs": f"Export growth (asinh), {YEAR_T0}-{YEAR_T1}",
        "density_sd": f"Relatedness density, {YEAR_T0} (sd)",
        "ihs_x0": f"Initial exports, {YEAR_T0} (asinh)",
        "loc": "Country",
        "prod": "Product",
    },
    notes=(
        "Columns 1 to 3 are linear probability models, column 4 is OLS. "
        "Standard errors clustered by country."
    ),
).make("gt")

In [ ]:
# The pooled intercept in the table (about 5.2%) is the fitted rate at the
# FULL-frame mean density; candidates average below that mean, which is why the
# base rate printed here is a little lower.
b_pool = float(m_entry_pooled.coef()["density_sd"])
b_entry = float(m_entry.coef()["density_sd"])
b_exit = float(m_exit.coef()["density_sd"])
b_growth = float(m_growth.coef()["density_sd"])
base_entry = frame.query("incumbent == 0")["appeared"].mean()
print(
    f"pooled: one sd of density -> entry probability {b_pool * 100:+.1f}pp on a "
    f"base rate of {base_entry:.1%}, multiplying the average entry rate by "
    f"roughly {1 + b_pool / base_entry:.1f}"
)
_inc = frame.query("incumbent == 1").copy()
_exit_dec = _inc.groupby(pd.qcut(_inc["density"], 10, labels=False))["exited"].mean()
print(
    f"exits mirror it: {_exit_dec.iloc[0]:.0%} of the least-related incumbents left the "
    f"basket, {_exit_dec.iloc[-1]:.0%} of the most-related"
)

In [ ]:
# Self-check: signs and sanity, not memorized coefficients.
assert b_pool > 0 and 0.005 < b_pool < 0.15, (
    f"the pooled entry effect per sd (got {b_pool:.3f}) should be a few "
    "percentage points, in line with the bar chart it summarizes."
)
assert b_entry > 0 and abs(m_entry.tstat()["density_sd"]) > 3, (
    "entry should rise with lagged density even after absorbing country and "
    "product averages; a sign flip means the density or event columns are inverted."
)
assert b_exit < 0, (
    "exit should FALL with lagged density: products near the basket survive. "
    "A positive sign suggests the exit definition is flipped."
)
assert b_growth > 0, (
    "surviving exports should also GROW faster where density was high; check "
    "that growth_ihs is final minus initial, not the reverse."
)
assert b_entry > b_pool, (
    "the within estimate should exceed the pooled one here: the fixed effects "
    "strip out country and product averages that dilute the pooled slope."
)
print("OK: density predicts entry (+), exit (-), growth (+).")

Read the table columns against each other. Column 1 is the headline number: across the whole world, one standard deviation of density buys a few percentage points of entry probability on a small base rate, the bar chart compressed into one coefficient. Column 2 asks a sharper question, whether the same country enters the same product more often when it sits unusually close to it, and the per-unit coefficient jumps. That is not a contradiction. The fixed effects remove the parts of density that vary between countries and between products, and what remains, the variation the fixed effects leave behind, is small but highly informative, so a coefficient per standard deviation of the overall density now describes a move far larger than any cell actually makes within the fixed effects (a typical within move is under a tenth of an overall standard deviation, which takes the within coefficient straight back to the pooled magnitude). The lesson for reading any fixed-effects table: check what variation identifies the coefficient before translating it into policy magnitudes, and take magnitudes from the design whose variation matches your question.

The specification is the compact classroom version of the morning's. Lecture 5's tables use a probit for the extensive margin (whether a product appears at all) and add a second density built from country similarity; the stretch lets you rerun ours as a logit and see how little the conclusion moves.

### Interpret (no AI)

Write two sentences for a policy audience stating the entry result in plain words, using your own numbers above: the base entry rate and the effect of one standard deviation of density. Your sentences must be ones the regression can actually defend, so mind the difference between "products with higher density were entered more often" and "raising density makes entry happen". One of those two is not supported by anything we ran today.

TODO(concept): your answer here.

## 8. What it means for Morocco

### Predict before running

Name two products you believe your country started exporting competitively since 2012. Write them down. Then run the cell and look for them, or their family, in the list.

In [ ]:
# The country's actual entries, biggest first, with the density each carried at t0.
ma_cand = cand.query("loc == @COUNTRY").set_index("prod")
ma_entries = (
    ma_cand.query("appeared == 1")
    .assign(
        product=lambda d: d.index.map(prod_name),
        exports_t1_musd=lambda d: (d["x1"] / 1e6).round(0),
        density_pctile=lambda d: (ma_cand["density"].rank(pct=True).reindex(d.index) * 100).round(
            0
        ),
    )
    .sort_values("x1", ascending=False)
    .loc[:, ["product", "exports_t1_musd", "density_pctile"]]
)
print(f"{COUNTRY}: {len(ma_entries)} entries between {YEAR_T0} and {YEAR_T1}")
ma_entries.head(12)

In [ ]:
# Where did the entered products rank, at t0, among ALL of the country's
# candidate products? A mean percentile of 50 would mean entries were random
# draws from the candidate list; the Principle of Relatedness predicts higher.
entry_pctiles = ma_cand["density"].rank(pct=True)[ma_entries.index] * 100
mean_pct = entry_pctiles.mean() if len(ma_entries) else float("nan")
_entry_x1 = ma_cand.loc[ma_entries.index, "x1"]
weighted_pct = (entry_pctiles * _entry_x1).sum() / _entry_x1.sum()
below_median_share = _entry_x1[entry_pctiles < 50].sum() / _entry_x1.sum()
print(
    f"average t0 density percentile of {COUNTRY}'s entries, among its own "
    f"candidates: {mean_pct:.0f} (50 would mean entries were random draws)"
)
print(
    f"weighted by {YEAR_T1} export value the average falls to {weighted_pct:.0f}, and "
    f"{below_median_share:.0%} of entry value came from below-median-density products"
)

In [ ]:
# Self-check: the entries concentrate where density was high.
assert len(ma_entries) > 0, (
    "no entries found for the focus country; check the COUNTRY code and window."
)
assert mean_pct > 55, (
    f"the mean density percentile of actual entries (got {mean_pct:.0f}) should "
    "sit clearly above 50: that is the Principle of Relatedness in one number."
)
print(f"OK: {COUNTRY}'s entries averaged the {mean_pct:.0f}th density percentile.")

For Morocco the list holds two stories, and the percentile column separates them. The organic one is food, plastics, and metals: Tropical Fruits, Sugar, Plastic Pipes, Other Steel Articles entered from the 80th to 100th density percentile, exactly where the principle says entries come from, and across all 33 entries the mean percentile of 69 is the Principle of Relatedness in one number. The automotive cluster is the second story: Vehicle Parts, Lighting, Bodies, and Engine Parts entered from the 24th to 49th percentile, further out than organic diversification usually reaches, on the back of a decade of deliberate foreign direct investment attraction around the Renault and later Stellantis plants. That is the "strategic bets" cell of this morning's policy table, executed. Both stories are real, and the second is the warning against reading the average as destiny: the regression describes where entries usually come from, and policy occasionally carries a country across the distribution. The value-weighted print above makes the same point in one number, since counting each entry by its size pulls the average percentile down toward the automotive bets.

The same evidence now points forward. The pooled regression from section 7 maps any density to an expected entry rate, so we can score today's candidates with the model fitted on the past window: if the next decade behaves like the last one, this is the base rate your shortlist is drawing from.

In [ ]:
# Today's candidates, scored with the pooled entry model fitted on the past
# window. The model was fitted on density in sd units, so today's densities go
# through the SAME standardization (the t0 frame's mean and sd) before scoring.
today = res_t1["cdata"].query("loc == @COUNTRY").set_index("prod")
today = today[res_t1["mcp"].loc[COUNTRY] == 0][["density", "pci"]].dropna()
today["density_sd"] = (today["density"] - frame["density"].mean()) / frame["density"].std()
# A linear probability model can dip below zero far from the mean; clip at zero
# and read tiny values as "historically rare".
today["expected_entry_rate"] = np.clip(
    m_entry_pooled.predict(newdata=today.reset_index()), 0, None
)
pci_median = today["pci"].median()
outlook = (
    today.query("pci > @pci_median")
    .nlargest(12, "density")
    .assign(product=lambda d: d.index.map(prod_name))
    .loc[:, ["product", "density", "pci", "expected_entry_rate"]]
    .round({"density": 3, "pci": 2, "expected_entry_rate": 3})
)
outlook

In [ ]:
# Self-check: the forward scoring is what it claims.
assert today["expected_entry_rate"].between(0, 1).all(), (
    "expected entry rates are probabilities and must lie in [0, 1] after the "
    "clip; wildly negative raw predictions mean the standardization used the "
    "wrong mean or sd."
)
assert outlook["expected_entry_rate"].notna().all(), (
    "every shortlisted candidate needs a score; missing values mean the predict "
    "call did not see the density_sd column."
)
_top_rate = today.nlargest(1, "density")["expected_entry_rate"].iloc[0]
assert _top_rate == today["expected_entry_rate"].max(), (
    "the model is linear in density, so the highest-density candidate must "
    "carry the highest expected rate. If not, the scoring reused the wrong model."
)
print(f"OK: {len(outlook)} candidates scored with the fitted pooled model.")

Yesterday's table said which products are near. Today's column says how often products that close to a basket actually got entered, on average, across the whole world's experience of the last decade. Read the level as the message: even your best candidates carry single-digit expected rates over a full decade, so diversification is a portfolio problem, and which specific product lands is exactly what the data cannot tell you. Section 11 says so out loud.

## 9. Countries: complexity and growth

The morning's growth regression made the same argument one level up: complexity predicts which countries grow. The mechanism is the week's story in reverse, since a complex basket signals many capabilities, and many capabilities mean many nearby products to expand into. We close the loop by running that regression on our own numbers, in the shape of Lecture 5's table: annualized growth of GDP per capita over a decade window, explained by initial income, a natural-resource control, initial ECI, and initial COI, pooled over two windows with a window fixed effect. One deviation from the lecture's table is forced by our data: it controls for the change in net resource exports, and a packet with exports only cannot compute net, so we control for the initial resource share of exports instead. Expect that control to behave differently from the morning's.

### Predict before running

Initial income enters with a negative sign in every growth regression since the 1990s (poorer countries grow faster, conditionally). The question is whether complexity survives next to it: once you hold initial income and resource wealth fixed, does a one standard deviation edge in ECI still buy measurable growth? Commit to yes or no, and to a magnitude in percentage points per year.

In [ ]:
def zscore(s: pd.Series) -> pd.Series:
    """Standardize to mean 0, sd 1, as on Days 1 and 2."""
    return (s - s.mean()) / s.std()


res_early = complexity_for_year(YEAR_EARLY)
gdp_pc = units.pivot_table(index="Unit", columns="Period", values="GDP PC")

rows = []
for res0, (t0, t1) in [(res_early, (YEAR_EARLY, YEAR_T0)), (res_t0, (YEAR_T0, YEAR_T1))]:
    X0 = res0["X"]
    nr_ids = [c for c in X0.columns if c[4:6] in ("25", "26", "27")]  # HS 25-27: minerals, fuels
    w = pd.DataFrame(
        {
            "gdp0": gdp_pc[t0],
            "gdp1": gdp_pc[t1],
            "eci0": zscore(res0["cdata"].groupby("loc")["eci"].first()),
            "coi0": zscore(res0["cdata"].groupby("loc")["coi"].first()),
            "nr_share0": X0[nr_ids].sum(axis=1) / X0.sum(axis=1),
        }
    ).dropna()
    w = w[(w["gdp0"] > 0) & (w["gdp1"] > 0)]
    w["years"] = t1 - t0
    w["window"] = f"{t0}-{t1}"
    rows.append(w)
gframe = pd.concat(rows).rename_axis("loc").reset_index()
gframe["ln_gdp0"] = np.log(gframe["gdp0"])

# TODO(core): annualized log growth of GDP per capita over each window.
#
# PROMPT: the log difference between gdp1 and gdp0, divided by the window length
# in years (the years column).

# TODO(core): write your code here
raise NotImplementedError()

print(f"{len(gframe)} country-window observations across {gframe['window'].nunique()} windows")

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. Annualized growth is (ln of final GDP pc minus ln of initial GDP pc) divided by the number of years, so a value of 0.02 reads as 2 percent per year.

Hint 2. `(np.log(gframe["gdp1"]) - np.log(gframe["gdp0"])) / gframe["years"]`, one vectorized line.
</details>

In [ ]:
m_g1 = pf.feols("growth_ann ~ ln_gdp0 + eci0 | window", data=gframe, vcov={"CRV1": "loc"})
m_g2 = pf.feols(
    "growth_ann ~ ln_gdp0 + nr_share0 + eci0 | window", data=gframe, vcov={"CRV1": "loc"}
)
m_g3 = pf.feols(
    "growth_ann ~ ln_gdp0 + nr_share0 + eci0 + coi0 | window",
    data=gframe,
    vcov={"CRV1": "loc"},
)

ETable(
    [m_g1, m_g2, m_g3],
    labels={
        "growth_ann": "Annualized GDP pc growth",
        "ln_gdp0": "Initial GDP pc (log)",
        "nr_share0": "Natural-resource export share",
        "eci0": "Initial ECI (sd)",
        "coi0": "Initial COI (sd)",
        "window": "Decade window",
    },
    notes=(
        "Country-window observations, two windows pooled with window fixed "
        "effects. Standard errors clustered by country."
    ),
).make("gt")

In [ ]:
b_eci = float(m_g2.coef()["eci0"])
print(f"one sd of initial ECI -> {b_eci * 100:+.2f}pp of annual growth, income held fixed")
print(f"corr(initial ECI, initial COI) = {gframe[['eci0', 'coi0']].corr().iloc[0, 1]:.2f}")

In [ ]:
# Self-check: the two relationships every growth regression must show here.
assert float(m_g2.coef()["ln_gdp0"]) < 0, (
    "conditional convergence: richer countries grow slower once complexity is "
    "held fixed. A positive sign points at a units problem in gdp or growth."
)
assert b_eci > 0 and abs(m_g2.tstat()["eci0"]) > 2, (
    "initial ECI should predict growth clearly in the pooled two-window design. "
    "If it does not, check that eci0 comes from the WINDOW-START year."
)
assert 250 < len(gframe) < 450, (
    f"expected roughly 350 country-window observations (got {len(gframe)}); a "
    "much smaller frame means the GDP join dropped countries it should not have."
)
print(f"OK: convergence holds and complexity predicts growth (+{b_eci * 100:.2f}pp/sd/yr).")

The lecture's version of this table pools three decades of data and adds more controls, and its coefficients are in the same range: a standard deviation of initial complexity is worth a bit under one percentage point of annual growth. Read column 3 with the printed ECI-COI correlation in mind: the two indices largely measure the same underlying position, so when both enter they split shared variance and ECI's coefficient weakens, and the headline figure is the ECI-only baseline of column 2. Over a decade that compounds to a difference most ministries would fight for. Also try the single-window version at home by dropping the first window from `rows`: with half the observations the ECI coefficient keeps its size and loses precision, a small lesson in why the lecture pools decades.

The section's last exhibit places every country on the two axes the morning called the strategic setting: current complexity (ECI) against how well positioned the country is to gain more (COI). The quadrant names are the lecture's.

In [ ]:
q = pd.DataFrame(
    {
        "eci": zscore(res_t1["cdata"].groupby("loc")["eci"].first()),
        "coi": zscore(res_t1["cdata"].groupby("loc")["coi"].first()),
    }
).dropna()

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(q["coi"], q["eci"], s=16, alpha=0.35, color="#5b9bd5")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
for code in [COUNTRY, *COMPARATORS]:
    if code in q.index:
        ax.scatter(
            q.loc[code, "coi"],
            q.loc[code, "eci"],
            s=110,
            color="#ee3e4c" if code == COUNTRY else "#e5bd4f",
            zorder=5,
            edgecolor="black",
            linewidth=0.8,
        )
        ax.annotate(
            code,
            (q.loc[code, "coi"], q.loc[code, "eci"]),
            xytext=(6, 6),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold",
        )
quad_style = dict(fontsize=9, color="#555555", style="italic", ha="center")
ax.text(0.98, 0.98, "Let it be:\nit ain't broke", transform=ax.transAxes, va="top", **quad_style)
ax.text(
    0.02,
    0.98,
    "Hey Jude, make it better:\ncompetitiveness policy",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=9,
    color="#555555",
    style="italic",
)
ax.text(
    0.98,
    0.02,
    "Stairway to heaven:\nparsimonious industrial policy",
    transform=ax.transAxes,
    va="bottom",
    ha="right",
    fontsize=9,
    color="#555555",
    style="italic",
)
ax.text(
    0.02,
    0.02,
    "Bridge over troubled waters:\nstrategic bets",
    transform=ax.transAxes,
    va="bottom",
    ha="left",
    fontsize=9,
    color="#555555",
    style="italic",
)
ax.set_xlabel(f"Complexity Outlook Index, {YEAR_T1} (sd)")
ax.set_ylabel(f"ECI, {YEAR_T1} (sd)")
ax.set_title("The strategic setting: where you are vs where you can reach")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

_ma_quad = (
    ("high" if q.loc[COUNTRY, "eci"] > 0 else "low")
    + " ECI, "
    + ("high" if q.loc[COUNTRY, "coi"] > 0 else "low")
    + " COI"
)
print(f"{COUNTRY}: {_ma_quad}")

In [ ]:
# Self-check: the quadrants partition the world.
_counts = [
    ((q["eci"] > 0) & (q["coi"] > 0)).sum(),
    ((q["eci"] > 0) & (q["coi"] <= 0)).sum(),
    ((q["eci"] <= 0) & (q["coi"] > 0)).sum(),
    ((q["eci"] <= 0) & (q["coi"] <= 0)).sum(),
]
assert sum(_counts) == len(q), "every country must land in exactly one quadrant."
assert COUNTRY in q.index, "the focus country must appear on the chart."
assert q[["eci", "coi"]].corr().iloc[0, 1] > 0.3, (
    "ECI and COI should correlate positively (complex baskets sit near complex "
    "products); near-zero correlation points at a merge problem."
)
print(f"OK: quadrants partition {len(q)} countries; {COUNTRY} is {_ma_quad}.")

Read the chart against the morning's strategic-setting grid. Morocco sits just under the world complexity midline with a clearly above-average outlook, in the corner where the lecture prescribes parsimonious industrial policy: many trees nearby, so helping short jumps goes far. Korea illustrates the top of the ladder: the highest complexity on the chart, with an outlook no better than the world average, since a country already exporting most complex products has little left to reach by moving nearby, the lecture's "many letters, hard to add more" case. Two cautions before this chart enters a slide deck: positions come from our trade-only, single-year pipeline, so borderline placements (Morocco sits barely below the ECI midline) mark a neighborhood, so read them loosely, and the quadrant labels carry strategy vocabulary with no measurement behind the exact boundaries.

## 10. ECI over time: freeze the yardstick

Section 9 used each window's initial ECI, which raises a question every country team will want to answer on Friday: has the country's complexity risen over the last two decades? Comparing raw ECI values across years cannot answer it. Each year's index is estimated from that year's cross-section alone and standardized to mean 0 and standard deviation 1 among that year's countries, and the product scores behind it are re-estimated every year too, so the basket and the yardstick move at the same time. Growth Lab practice separates the two with a trick: freeze every product's PCI at its most recent estimate, then price each year's basket at those fixed scores. For that to be legitimate we need one property of the index, and deriving it also settles which weights are the right ones.

### The property, derived

In the binary world, a country's ECI equals the plain average of the PCI of the products it exports with RCA above 1. Our pipeline is continuous, so the question is whether ECI is still a weighted average of PCI, and with which weights. The natural guess, weighting each product by its RCA, turns out to be wrong.

Follow the package's construction, which is Day 1's with continuous weights. It first estimates the raw product score $q_p$, the eigenvector for the second-largest eigenvalue of the product-product matrix $\widetilde{M}_{pp'}$, now assembled from the presence weights $\widehat{M}_{cp} = \mathrm{RCA}_{cp} / (1 + \mathrm{RCA}_{cp})$. It then defines the raw country score as the $\widehat{M}$-weighted average of $q$ over the country's basket:

$$ k_c = \frac{\sum_p \widehat{M}_{cp} \, q_p}{\sum_p \widehat{M}_{cp}} $$

The step everything hangs on comes last. The package standardizes both vectors with the same two constants, the mean $\mu_k$ and standard deviation $\sigma_k$ of the raw country score:

$$ \mathrm{ECI}_c = \frac{k_c - \mu_k}{\sigma_k}, \qquad \mathrm{PCI}_p = \frac{q_p - \mu_k}{\sigma_k} $$

A weighted average passes through that rescaling untouched, since every term shifts by the same $\mu_k$ and divides by the same $\sigma_k$:

$$ \frac{\sum_p \widehat{M}_{cp} \, \mathrm{PCI}_p}{\sum_p \widehat{M}_{cp}} = \frac{k_c - \mu_k}{\sigma_k} = \mathrm{ECI}_c $$

So the binary property survives in the continuous world, with two corrections. The weights are $\widehat{M}$, whose cap below 1 does real work: raw RCA is unbounded, so a single hyper-specialized export with an RCA in the hundreds would drag the whole average toward its own PCI. And the denominator is the sum of the weights, where the binary version divides by a product count. One convention makes the identity exact rather than approximate: py-ecomplexity standardizes PCI with the ECI's constants, the convention the original Stata implementation set. A package that standardizes PCI by its own mean and standard deviation, as R's `economiccomplexity` does, keeps the identity up to a linear rescaling, and the R companion shows the one-line recalibration.

In [ ]:
# Both ingredients come from the section 1 results: the package's PCI (one value
# per product) and ECI (one value per country) for YEAR_T1.
pci_frozen = res_t1["cdata"].groupby("prod")["pci"].first()
eci_t1 = res_t1["cdata"].groupby("loc")["eci"].first()
m_hat_t1 = res_t1["m_hat"]

# TODO(core): the derivation's last display, in code. Multiply each country's
# row of weights by the frozen product scores, sum across products, and divide
# by the sum of the weights. Pandas aligns pci_frozen on the columns, so this
# is one line on the matrix, no loops.
#
# PROMPT: define eci_check as the M-hat-weighted average of pci_frozen. Until
# it exists, the cell stops with a NameError.

# TODO(core): complete the one missing line below.

gap = (eci_check - eci_t1).abs().max()
print(f"max |weighted average - package ECI| across {len(eci_check)} countries: {gap:.1e}")

# The wrong weights, for contrast: raw RCA is unbounded, so one big bet drowns
# out the rest of the basket and the average drifts far from the index.
eci_rca_w = (res_t1["rca"] * pci_frozen).sum(axis=1) / res_t1["rca"].sum(axis=1)
print(
    f"raw-RCA-weighted version: correlation {eci_rca_w.corr(eci_t1):.2f} with the "
    f"index, biggest gap {(eci_rca_w - eci_t1).abs().max():.1f} standard deviations"
)
print(
    f"{COUNTRY}: package ECI {eci_t1[COUNTRY]:+.3f} | M-hat weights "
    f"{eci_check[COUNTRY]:+.3f} | raw RCA weights {eci_rca_w[COUNTRY]:+.3f}"
)

<details>
<summary><b>Stuck? Reveal a hint (then a fuller hint).</b></summary>

Hint 1. `m_hat_t1 * pci_frozen` multiplies every country's row of weights by the product scores, because pandas aligns a Series on the DataFrame's columns. You then need a sum across products (`axis=1`) on top, divided by the same sum applied to the weights alone.

Hint 2. `eci_check = (m_hat_t1 * pci_frozen).sum(axis=1) / m_hat_t1.sum(axis=1)`.
</details>

In [ ]:
# Self-check: the identity holds to numerical precision, not approximately.
assert np.allclose(eci_check, eci_t1.reindex(eci_check.index), atol=1e-8), (
    "the M-hat-weighted average of PCI must reproduce the package ECI exactly. "
    "The two usual causes: the weights must be M-hat, not raw RCA, and the "
    "denominator must be the sum of the weights, not the number of products."
)
assert eci_check.between(pci_frozen.min(), pci_frozen.max()).all(), (
    "a weighted average must lie between the smallest and largest PCI; a value "
    "outside that range means the division by the summed weights went missing."
)
assert eci_rca_w.corr(eci_t1) < 0.999, (
    "the raw-RCA version should visibly drift from the index; if it matches, "
    "eci_rca_w was probably computed from M-hat too."
)
print(f"OK: ECI is the M-hat-weighted average of PCI (largest gap {gap:.1e}).")

The identity makes the time series cheap. Freeze PCI at the latest year, rebuild only the first half of the pipeline for every year in the packet, exports through M-hat, and take the weighted average each year. No eigenvector is re-estimated after this point, and that is the entire point of the trick: the product scores stay put, so whatever movement you see is the basket changing, and only that.

### Predict before running

Morocco's 2001 basket against its 2023 basket, priced at the same frozen scores: did average complexity rise, fall, or hold flat? Same question for Korea, which already sat near the top of the world distribution in 2001. Commit to both answers before running the loop.

In [ ]:
def mhat_for_year(year: int) -> pd.DataFrame:
    """Exports through M-hat for one year: the cheap half of complexity_for_year."""
    year_long = exports.query("Period == @year")[["Unit", "Field ID", "Outputs"]]
    X = year_long.pivot_table(index="Unit", columns="Field ID", values="Outputs", fill_value=0.0)
    X = X.loc[X.sum(axis=1) > 0, X.sum(axis=0) > 0]
    shares = X.div(X.sum(axis=1), axis=0)
    world_share = X.sum(axis=0) / X.sum().sum()
    rca = shares.div(world_share, axis=1)
    return rca / (1 + rca)


years = sorted(exports["Period"].unique())
frozen_rows = []
coverage = {}
for year in years:
    m_hat_y = mhat_for_year(year)
    # Only products that carry a frozen score can enter the average. This packet
    # keeps the product list stable across years, so nothing is dropped; the
    # self-check verifies that instead of assuming it.
    scored = m_hat_y.columns.intersection(pci_frozen.index)
    w = m_hat_y[scored]
    coverage[year] = float((w.sum(axis=1) / m_hat_y.sum(axis=1)).min())
    eci_frozen = (w * pci_frozen[scored]).sum(axis=1) / w.sum(axis=1)
    frozen_rows.append(
        pd.DataFrame({"year": year, "loc": eci_frozen.index, "eci_frozen": eci_frozen.values})
    )
eci_series = pd.concat(frozen_rows, ignore_index=True)
print(
    f"frozen-PCI series: {len(years)} years ({years[0]} to {years[-1]}), "
    f"{eci_series['loc'].nunique()} countries appear at least once"
)

In [ ]:
# Self-check: the frozen year must land back on the package index, and every
# year's average must be priced on essentially the full basket.
_frozen_t1 = eci_series.query("year == @YEAR_T1").set_index("loc")["eci_frozen"]
assert np.allclose(_frozen_t1, eci_t1.reindex(_frozen_t1.index), atol=1e-8), (
    f"in {YEAR_T1} the series must equal the package ECI. If the one-year check "
    "above passed and this fails, the loop aligns or weights differently."
)
assert min(coverage.values()) > 0.99, (
    "in some year more than 1% of a country's basket weight sits on products "
    "with no frozen score, so the average would be priced on a truncated basket."
)
assert eci_series["eci_frozen"].between(pci_frozen.min(), pci_frozen.max()).all(), (
    "every value is a weighted average of PCIs and must stay inside the PCI range."
)
print(f"OK: {YEAR_T1} reproduces the index; minimum basket coverage {min(coverage.values()):.1%}.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
# The world as context: the middle 80% of countries, on the same frozen yardstick.
band = eci_series.groupby("year")["eci_frozen"].quantile([0.10, 0.90]).unstack()
ax.fill_between(band.index, band[0.10], band[0.90], color="#ededed", zorder=0)
ax.text(
    years[0] + 0.3,
    band[0.10].iloc[0] + 0.12,
    "world, 10th to 90th percentile",
    fontsize=8,
    color="#999999",
)
ax.axhline(0, color="grey", lw=0.8)
palette = ["#5b9bd5", "#e5bd4f", "#666666"]
for code, color in [(COUNTRY, "#ee3e4c")] + list(zip(COMPARATORS, palette)):
    s = eci_series.query("loc == @code").set_index("year")["eci_frozen"].sort_index()
    ax.plot(s.index, s.values, color=color, lw=2.6 if code == COUNTRY else 1.6)
    ax.annotate(
        code,
        (s.index[-1], s.iloc[-1]),
        xytext=(6, -3),
        textcoords="offset points",
        color=color,
        fontsize=10,
        fontweight="bold",
    )
ax.set_xlim(years[0], years[-1] + 2)
ax.set_xlabel("Year")
ax.set_ylabel(f"ECI with product scores frozen in {YEAR_T1}")
ax.set_title("The basket moves, the yardstick does not")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

for code in [COUNTRY, *COMPARATORS]:
    s = eci_series.query("loc == @code").set_index("year")["eci_frozen"]
    print(
        f"{code}: {s.get(years[0], float('nan')):+.2f} in {years[0]} "
        f"-> {s.get(YEAR_T1):+.2f} in {YEAR_T1}"
    )

Morocco's basket, priced at fixed product scores, moved from about a third of a standard deviation below the frozen-year world average in 2001 to just under that average in 2023, so the diversification documented in section 8 reads as a real compositional upgrade. Korea keeps climbing even from the top of the distribution, visible here because nothing re-standardizes each year back to mean zero. Keep two caveats attached before this chart enters a slide deck. The exercise prices every basket at one year's scores, so in the frozen year the series equals the published index exactly (that is the self-check), and in every other year it is deliberately a different object from that year's own ECI, blind to any product whose true complexity changed over the period; freezing PCI in a different year and re-drawing the chart is a cheap robustness check for Friday. And the levels inherit Day 1's caveat: a trade-only, single-eigenvector index supports trajectories and gaps better than precise country positions.

## 11. What today's evidence does and does not say

The regressions are descriptive. Say that precisely, because the description is the whole claim. Density is not a policy lever; it moves when the basket moves. What the entry regression shows is that the capabilities already visible in a country's basket tell you where its next products came from, historically, at both the product level and the country level. Nothing today identifies what happens if a government pushes a product it is not near: Morocco's automotive bet shows such pushes can succeed, and the regression cannot tell you when they do. The growth regression inherits the same limit at the country grain, since initial ECI travels with income, institutions, and everything else a capable state does, so its coefficient is a conditional correlation and says nothing about what deliberately building complexity would do to growth. The lag removes the mechanical part of the circularity (entering a product raises density around it), and it cannot remove the deeper one: unobserved capabilities drive both density and entry. On top of that, everything inherits Day 1's caveats, since RCA > 1 is a knife edge, exports are not production, and services and the informal economy are invisible here.

### Interpret (no AI)

Below is a summary of today's session written by a chatbot. Mark every claim the analysis cannot support, then rewrite it in two sentences a minister could quote safely.

> "Our regression demonstrates that relatedness density is a key driver of export diversification: raising a product's density by one standard deviation causes an increase of several percentage points in its probability of entry. Morocco should therefore prioritize investments that increase density around strategic sectors. Since the growth regression shows that complexity raises GDP growth, these interventions will also accelerate national growth."

TODO(concept): list the overclaims, then your rewrite.

## 12. Recall: close the loop

From memory, without scrolling up:

1. Define an entry event precisely. Which two conditions, in which two years?
2. Why must density be measured in the earlier year? Name the bug you fixed.
3. What do the country and product fixed effects absorb, and name one confounder they cannot remove.
4. What pattern in the bar chart would have falsified the Principle of Relatedness?
5. What does the growth regression add that the entry regression alone cannot say?
6. ECI is a weighted average of PCI in our continuous pipeline. Which weights, which denominator, and which standardization convention make that identity exact?

TODO(concept): your answers here.

## Stretch (for fast finishers, scaffolding removed)

(a) The logit. The linear probability model is transparent and slightly wrong for a rare outcome. Rerun the entry regression as `pf.feglm("appeared ~ density_sd | loc + prod", data=..., family="logit")` and translate the coefficient into an effect on the entry probability at the base rate. How different is the story?

(b) Pool the windows. Build the same entry frame for the 2001 to 2012 window, stack it with the 2012 to 2023 one, add a window fixed effect, and see what pooling does to the precision. The growth regression in section 9 already uses this design.

(c) Break the circularity harder. Density is computed from a proximity matrix that the country's own exports helped estimate. Recompute proximity using only half the countries (drawn at random), compute density for the other half from it, and rerun the entry regression on the held-out half.

(d) The residual approach. Lecture 5's two-stage method uses the residual from section 3's cross-sectional regression (the gap between what the ecosystem implies and what the country actually does) to predict growth. Extract the residuals from `m_cross` and check whether cells that underperform their ecosystem (large negative residual) grow faster to t1.

In [ ]:
# (Your stretch code here, no scaffold, no self-check.)